In [1]:
!pip install -q transformers datasets peft accelerate bitsandbytes trl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 31.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.6/564.6 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 106.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.3/564.3 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 43.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 80.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 92.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 77.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.

In [2]:
import json
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print(f" PyTorch version: {torch.__version__}")
print(f" CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f" GPU: {torch.cuda.get_device_name(0)}")
    print(f" VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

2025-10-11 12:12:20.586432: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760184740.832649      37 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760184740.902247      37 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


 PyTorch version: 2.6.0+cu124
 CUDA available: True
 GPU: Tesla T4
 VRAM: 15.83 GB


In [8]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
DATA_FILE = "/kaggle/input/data-askly/data_training.json"  
OUTPUT_DIR = "/kaggle/working/finetuned_model"

# LoRA Config
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.1

# Training Config
BATCH_SIZE = 8  # Kaggle GPU có 16GB, có thể dùng batch lớn
GRADIENT_ACCUMULATION = 2
LEARNING_RATE = 2e-4
EPOCHS = 3
MAX_LENGTH = 1024

print(f" Model: {MODEL_NAME}")
print(f" Data: {DATA_FILE}")
print(f" Output: {OUTPUT_DIR}")

 Model: Qwen/Qwen2.5-3B-Instruct
 Data: /kaggle/input/data-askly/data_training.json
 Output: /kaggle/working/finetuned_model


In [9]:
print(" Loading data...")
with open(DATA_FILE, 'r', encoding='utf-8') as f:
    data = json.load(f)
print(f" Loaded {len(data)} samples")
# Format data
formatted_data = []
for item in data:
    text = f"<|im_start|>user\n{item['question']}<|im_end|>\n<|im_start|>assistant\n{item['answer']}<|im_end|>"
    formatted_data.append({"text": text})
dataset = Dataset.from_list(formatted_data)
print(f" Dataset created: {len(dataset)} samples")

 Loading data...
 Loaded 1010 samples
 Dataset created: 1010 samples


In [10]:
print(" Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f" Tokenizer loaded")
print(f"   Vocab size: {tokenizer.vocab_size}")
print(f"   Pad token: {tokenizer.pad_token}")

# Tokenize function
def tokenize(examples):
    result = tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length"
    )
    result["labels"] = result["input_ids"].copy()
    return result

# Tokenize dataset
print(" Tokenizing dataset...")
tokenized_dataset = dataset.map(
    tokenize,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing"
)
print(f" Tokenization complete")

 Loading tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

 Tokenizer loaded
   Vocab size: 151643
   Pad token: <|endoftext|>
 Tokenizing dataset...


Tokenizing:   0%|          | 0/1010 [00:00<?, ? examples/s]

 Tokenization complete


In [11]:
print(" Loading model with 4-bit quantization...")

# 4-bit config for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print(" Model loaded")

# Prepare for training
model = prepare_model_for_kbit_training(model)

# LoRA config
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA
model = get_peft_model(model, lora_config)

print("LoRA applied")
model.print_trainable_parameters()


 Loading model with 4-bit quantization...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

 Model loaded
LoRA applied
trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


In [14]:
print(" Setting up training...")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,

    # Giảm batch size để vừa VRAM (T4 hoặc 3050)
    per_device_train_batch_size=1,
    gradient_accumulation_steps=max(16, GRADIENT_ACCUMULATION),  # bù hiệu batch lớn
    learning_rate=LEARNING_RATE,
    fp16=True,
    bf16=False,
    group_by_length=True,
    dataloader_pin_memory=False,
    dataloader_drop_last=False,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    logging_steps=20,
    save_steps=200,
    save_total_limit=2,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    # Không cần báo cáo ra WandB hoặc Hub
    report_to="none",
    eval_strategy="no",  # tắt eval giữa chừng để tránh OOM
)

# Một vài cấu hình thêm nên có (ở phần model)
model.config.use_cache = False  # cần cho gradient checkpointing
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
try:
    model._attn_implementation = "sdpa"  # nếu có PyTorch >=2.1
except Exception:
    pass

# Data collator: không pad thừa
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print("Starting training...")
total_steps = len(tokenized_dataset) // (1 * max(16, GRADIENT_ACCUMULATION)) * EPOCHS
print(f"   Total steps: {total_steps}")
print(f"   Estimated time: ~{(len(tokenized_dataset) * EPOCHS / 1000):.1f}-{(len(tokenized_dataset) * EPOCHS / 500):.1f} hours")

# Train
trainer.train()

print("Training completed!")


 Setting up training...
Starting training...
   Total steps: 189
   Estimated time: ~3.0-6.1 hours


Step,Training Loss
20,2.110000
40,1.394700
60,1.252700
80,0.991700
100,0.899100
120,0.825000
140,0.643000
160,0.537800
180,0.547100


Training completed!


In [15]:
print(" Saving model...")

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f" Model saved to: {OUTPUT_DIR}")

 Saving model...
 Model saved to: /kaggle/working/finetuned_model


In [19]:
print(" Testing model...")

# Chuẩn bị tokenizer/model cho suy luận
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model.eval()
if hasattr(model, "config"):
    model.config.use_cache = True  # suy luận nên bật cache

# Lấy danh sách EOS/STOP token hợp lệ (tự phát hiện theo model)
eos_ids = []
try:
    eot_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")  # Llama 3.x
    if eot_id is not None and eot_id != tokenizer.unk_token_id:
        eos_ids.append(eot_id)
except Exception:
    pass
if tokenizer.eos_token_id is not None:
    eos_ids.append(tokenizer.eos_token_id)
eos_ids = list(dict.fromkeys([i for i in eos_ids if i is not None]))  # unique & non-null
if not eos_ids:
    # fallback an toàn
    eos_ids = [tokenizer.eos_token_id] if tokenizer.eos_token_id is not None else None

def test_model(question, context: str = ""):
    messages = [
        {"role": "system", "content": "Bạn là trợ lý chỉ trả lời dựa trên ngữ cảnh; thiếu ngữ cảnh thì trả lời: 'không đủ thông tin'."},
        {"role": "user", "content": (f"Ngữ cảnh:\n{context}\n\nCâu hỏi: {question}" if context else question)},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.12,
            no_repeat_ngram_size=4,
            eos_token_id=eos_ids,
            pad_token_id=tokenizer.pad_token_id,
        )

    # Chỉ lấy phần mới sinh
    gen_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    return response

# Test
test_questions = [
    "Quy trình đăng ký học cải thiện điểm là gì?",
    "Học phí cải thiện bao nhiêu?",
    "Bằng tốt nghiệp Đại học FPT có ghi hình thức đào tạo không?",
]

print("\n" + "="*60)
for q in test_questions:
    print(f"\n Câu hỏi: {q}")
    ans = test_model(q)
    print(f"💬 Trả lời: {ans}")
    print("-"*60)


 Testing model...


 Câu hỏi: Quy trình đăng ký học cải thiện điểm là gì?
💬 Trả lời: Để đăng ký học lại hoặc học cải thiện, bạn vào FAP, mục 'Thủ tục/đơn từ' -> 'Register to improve mark (Làm đơn đăng ký cải thiện điểm)' và điền đầy đủ thông tin. Bạn nộp đơn trước khi học kỳ mới bắt đầu 1 tuần để được tính phí. Sau đó chờ kết quả. Nếu qua môn, bạn sẽ nhận điểm mới, nếu không qua sẽ giữ nguyên điểm cũ.
------------------------------------------------------------

 Câu hỏi: Học phí cải thiện bao nhiêu?
💬 Trả lời: Đối với các kỳ học đầu tiên, học phí được giảm 50%. Đối với các kỳ sau hoặc các môn có mức học phí cao hơn, bạn vẫn sẽ nhận được chính sách ưu đãi tương đương 30% - 40% học phí.
------------------------------------------------------------

 Câu hỏi: Bằng tốt nghiệp Đại học FPT có ghi hình thức đào tạo không?
💬 Trả lời: Có. Trên bảng điểm chi tiết của bằng cấp, bạn sẽ thấy mục 'Type of education' (Hình thức đào tạo) được ghi là 'TRÊN ĐẠT' hoặc 'TRÊM ĐẠT', cụ thể là bằng Cử nhân đ

In [17]:
print(" Creating zip file for download...")
import shutil
shutil.make_archive('/kaggle/working/finetuned_model', 'zip', OUTPUT_DIR)
print(" Model zipped: /kaggle/working/finetuned_model.zip")
print("   Click 'Output' tab bên phải để download")

 Creating zip file for download...
 Model zipped: /kaggle/working/finetuned_model.zip
   Click 'Output' tab bên phải để download
